In [ ]:
!pip install transformers datasets torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.


In [ ]:
import pandas as pd
from datasets import Dataset

In [ ]:
# Load your dataset
df = pd.read_csv('sentiment_yelp_data.csv')
df.head()

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,sentiment,sentiment_label
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,wife took birthday breakfast excellent weather...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0,Positive,2
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,idea people give bad reviews place goes show p...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0,Positive,2
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love gyro plate rice good also dig candy selec...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0,Positive,2
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,rosie dakota love chaparral dog park convenien...,review,uZetl9T0NcROGOyFfughhg,1,2,0,Positive,2
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,general manager scott petello good egg go deta...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0,Positive,2


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   business_id      10000 non-null  object
 1   date             10000 non-null  object
 2   review_id        10000 non-null  object
 3   stars            10000 non-null  int64 
 4   text             9999 non-null   object
 5   type             10000 non-null  object
 6   user_id          10000 non-null  object
 7   cool             10000 non-null  int64 
 8   useful           10000 non-null  int64 
 9   funny            10000 non-null  int64 
 10  sentiment        10000 non-null  object
 11  sentiment_label  10000 non-null  int64 
dtypes: int64(5), object(7)
memory usage: 937.6+ KB


In [ ]:
df.sentiment.unique()

array(['Positive', 'Negative', 'Neutral'], dtype=object)

In [ ]:
# Check for any missing or non-string values in the 'text' column
print(df['text'].isnull().sum())  # Check for null values
print(df['text'].apply(lambda x: isinstance(x, str)).sum())  # Check for non-string entries


1
9999


In [ ]:
# Drop rows where 'text' is null
df = df.dropna(subset=['text'])

# Ensure all values in the 'text' column are strings
df['text'] = df['text'].astype(str)


<ipython-input-31-3fa0f39c4241>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].astype(str)


In [ ]:
# Convert the dataset to a Hugging Face `Dataset`
# Ensure the `sentiment` column is mapped to integers (0 for negative, 1 for positive)
df['sentiment'] = df['sentiment'].map({'Negative': 0, 'Neutral': 1, 'Positive': 2})
dataset = Dataset.from_pandas(df)

# Peek at the dataset to ensure it's loaded correctly
print(dataset)

Dataset({
    features: ['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id', 'cool', 'useful', 'funny', 'sentiment', 'sentiment_label', '__index_level_0__'],
    num_rows: 9999
})


In [ ]:
#Tokenize data using BERT Tokenizer

In [ ]:
from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize function to apply to each example
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Apply the tokenizer to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)



/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/9999 [00:00<?, ? examples/s]

In [ ]:
#Prepare data for PyTorch Training

In [ ]:
# Remove unnecessary columns
tokenized_dataset = tokenized_dataset.remove_columns(['business_id', 'date','review_id', 'stars', 'type', 'user_id', 'cool', 'useful', 'funny', 'sentiment_label'])

# Rename the sentiment_label column to labels
tokenized_dataset = tokenized_dataset.rename_column('sentiment', 'labels')

# Set the dataset format to PyTorch tensors
tokenized_dataset.set_format('torch')



In [ ]:
#Split dataset to train and test


In [ ]:
# Split the dataset into training and testing sets (80% train, 20% test)
train_test_split = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(train_dataset)
print(test_dataset)


Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7999
})
Dataset({
    features: ['text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2000
})


In [ ]:
train_dataset['labels']

tensor([1, 1, 1,  ..., 0, 1, 0])

In [ ]:
#Load the model

In [ ]:
from transformers import BertForSequenceClassification

# Load pre-trained BERT for sequence classification (with 3 sentiment labels)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
#Define training parameters and initialize trainer

In [ ]:
from transformers import Trainer, TrainingArguments

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model
    evaluation_strategy="epoch",     # evaluate each epoch
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    num_train_epochs=1,              # number of epochs
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for logs
    logging_steps=10,
)

# Initialize the trainer
trainer = Trainer(
    model=model,                     # the pre-trained BERT model
    args=training_args,              # training arguments
    train_dataset=train_dataset,     # training dataset
    eval_dataset=test_dataset        # evaluation dataset
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
#Train Model / Fine-Tune BERT

In [ ]:
# Train the model
trainer.train()


Epoch,Training Loss,Validation Loss


In [ ]:
#Evaluate Model

In [ ]:
# Evaluate the model
results = trainer.evaluate()
print(results)


In [ ]:
#Save Model

In [ ]:
# Save the model and tokenizer
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')


In [ ]:
!zip mymodel.zip fine_tuned_model/

In [ ]:
#Using SavedModel

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification

# Load the saved tokenizer
tokenizer = BertTokenizer.from_pretrained('./fine_tuned_model')

# Load the saved model
model = BertForSequenceClassification.from_pretrained('./fine_tuned_model')

# Example text to classify sentiment
text = "The movie was amazing and I loved it!"

# Tokenize the input text (same as how it was done during training)
inputs = tokenizer(text, return_tensors='pt', padding='max_length', truncation=True)

import torch

# Perform inference (get the logits)
outputs = model(**inputs)

# Extract the predicted label (index of the maximum value in logits)
predictions = torch.argmax(outputs.logits, dim=1)

# Map the prediction to the actual label (e.g., positive, negative, neutral)
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}  # Adjust according to your label mapping
predicted_label = label_map[predictions.item()]

print(f"Predicted sentiment: {predicted_label}")
